In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

In [3]:
lista_macro = []

for filename in os.listdir(path='../../base_dados/brapi/macro'):
    
    # 2. Reconstruct the full absolute or relative path to the file
    full_path = os.path.join('../../base_dados/brapi/macro/',filename)
    
    # 3. Check if the current item is actually a file (and not a subfolder)
    # if os.path.isfile(full_path):
        
    #     # 4. Open and process the file safely
    #     with open(full_path, "r", encoding="utf-8") as file:
    #         content = file.read()
    #         print(f"--- Content of {filename} ---")
    #         print(pd.read_csv(full_path))
    dicio = {
        'macro':filename,
        'data':pd.read_csv(full_path)
    }

    lista_macro.append(dicio)

In [4]:
lista_macro[0]['data'].set_index('date', inplace=True)

In [5]:
lista_macro[0]['data'].describe().loc['std']['valor_anual']

np.float64(4.232295543007178)

In [6]:
selic_d = lista_macro[0]['data']

In [8]:
basedados_ativos = Path('../../base_dados/brapi/retornos/retornos.csv')
basedados_ibov = Path('../../base_dados/retorno_ibov_2015_2026.csv') 

lista_magic_formula = []

for filename in os.listdir(path='../../base_dados/brapi/magic_formula/'):
    
    # 2. Reconstruct the full absolute or relative path to the file
    full_path = os.path.join('../../base_dados/brapi/magic_formula/', filename)
    
    # 3. Check if the current item is actually a file (and not a subfolder)
    # if os.path.isfile(full_path):
        
    #     # 4. Open and process the file safely
    #     with open(full_path, "r", encoding="utf-8") as file:
    #         content = file.read()
    #         print(f"--- Content of {filename} ---")
    #         print(pd.read_csv(full_path))
    dicio = {
        'ativo':filename,
        'data':pd.read_csv(full_path)
    }

    lista_magic_formula.append(dicio)

df_ativos=pd.read_csv(basedados_ativos).set_index(['date']).fillna(0)

df_ibov=pd.read_csv(basedados_ibov).set_index(['Date']).fillna(0)



In [9]:
# lista_ativos=[]
# for ativo in df_ativos.columns.tolist():
#     lista_ativos.append(ativo.replace('.SA',''))
#     df_ativos.rename(columns={ativo:ativo.replace('.SA','')},inplace=True)

In [15]:
print(df_ativos.shape[0])
print(df_ibov.shape)


2863
(2849, 1)


In [12]:
print(f'tamanho mf original: ',len(lista_magic_formula))
lista_mf_check = []
lista_ativos_finais = []
for i in range(len(lista_magic_formula)):
    if lista_magic_formula[i]['ativo'] in df_ativos.columns.tolist():
        lista_ativos_finais.append(lista_magic_formula[i]['ativo'])
        dc = {
            'ativo':lista_magic_formula[i]['ativo'],
            'data':lista_magic_formula[i]['data']
        }
        lista_mf_check.append(dc)
    else:
        print(f"Nao está na lista: {lista_magic_formula[i]['ativo']}")
print(f'tamanho mf Final: ',len(lista_mf_check))
        

tamanho mf original:  78
tamanho mf Final:  78


In [13]:
lista_mf_check

[{'ativo': 'ABEV3',
  'data':     Unnamed: 0     endDate       ROC        EY
  0            0  2025-12-31  0.903659  0.111531
  1            1  2024-12-31  0.624135  0.141094
  2            2  2023-12-31  0.840339  0.101218
  3            3  2022-12-31  0.646090  0.093623
  4            4  2021-12-31  0.585286  0.089962
  5            5  2020-12-31  0.598088  0.085787
  6            6  2019-12-31  0.637351  0.073615
  7            7  2018-12-31  0.824850  0.092799
  8            8  2017-12-31  1.105830  0.069359
  9            9  2016-12-31  1.198593  0.096538
  10          10  2015-12-31  1.084847  0.104877
  11          11  2014-12-31  1.081966  0.102054
  12          12  2013-12-31  0.892491  0.099119
  13          13  2012-12-31  0.762046  0.000204},
 {'ativo': 'ALOS3',
  'data':     Unnamed: 0     endDate       ROC        EY
  0            0  2025-12-31  0.719754  0.069923
  1            1  2024-12-31  0.690710  0.089326
  2            2  2023-12-31  3.051214  0.259800
  3        

In [16]:
df_ativos = df_ativos[df_ativos.index.isin(df_ibov.index)]
print(df_ativos.shape)

(2847, 78)


In [17]:
df_ibov = df_ibov[df_ibov.index.isin(df_ativos[df_ativos.index.isin(df_ibov.index)].index)]
print(df_ibov.shape)

(2847, 1)


In [18]:
df_ativos = df_ativos.filter(lista_ativos_finais)
print(len(df_ativos))

2847


In [19]:
#ativos
df_ativos_2015 = df_ativos[df_ativos.index.str.startswith('2015')]
df_ativos_2016 = df_ativos[df_ativos.index.str.startswith('2016')]
df_ativos_2017 = df_ativos[df_ativos.index.str.startswith('2017')]
df_ativos_2018 = df_ativos[df_ativos.index.str.startswith('2018')]
df_ativos_2019 = df_ativos[df_ativos.index.str.startswith('2019')]
df_ativos_2020 = df_ativos[df_ativos.index.str.startswith('2020')]
df_ativos_2021 = df_ativos[df_ativos.index.str.startswith('2021')]
df_ativos_2022 = df_ativos[df_ativos.index.str.startswith('2022')]
df_ativos_2023 = df_ativos[df_ativos.index.str.startswith('2023')]
df_ativos_2024 = df_ativos[df_ativos.index.str.startswith('2024')]
df_ativos_2025 = df_ativos[df_ativos.index.str.startswith('2025')]
df_ativos_2026 = df_ativos[df_ativos.index.str.startswith('2026')]

#ibov
df_ibov_2015 = df_ibov[df_ibov.index.str.startswith('2015')]
df_ibov_2016 = df_ibov[df_ibov.index.str.startswith('2016')]
df_ibov_2017 = df_ibov[df_ibov.index.str.startswith('2017')]
df_ibov_2018 = df_ibov[df_ibov.index.str.startswith('2018')]
df_ibov_2019 = df_ibov[df_ibov.index.str.startswith('2019')]
df_ibov_2020 = df_ibov[df_ibov.index.str.startswith('2020')]
df_ibov_2021 = df_ibov[df_ibov.index.str.startswith('2021')]
df_ibov_2022 = df_ibov[df_ibov.index.str.startswith('2022')]
df_ibov_2023 = df_ibov[df_ibov.index.str.startswith('2023')]
df_ibov_2024 = df_ibov[df_ibov.index.str.startswith('2024')]
df_ibov_2025 = df_ibov[df_ibov.index.str.startswith('2025')]
df_ibov_2026 = df_ibov[df_ibov.index.str.startswith('2026')]

In [20]:
selic_d_2016 = selic_d[selic_d.index.isin(df_ativos_2016.index)]
selic_d_2016['valor_diario'] = selic_d_2016['valor_diario']/100

In [21]:
df_ativos_2016

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2016-01-04,-0.035857,0.0,-0.061592,-0.057285,-0.059519,-0.040421,-0.033928,-0.014629,-0.014515,-0.061646,...,-0.035836,0.0,-0.010953,-0.048801,-0.058083,-0.026085,0.0,-0.102648,-0.020728,0.0
2016-01-05,0.015689,0.0,-0.103491,0.022078,0.007087,0.038272,0.003523,0.007793,0.003637,0.063950,...,-0.015476,0.0,-0.004887,0.022957,-0.082191,-0.013400,0.0,0.001649,0.048486,0.0
2016-01-06,-0.009722,0.0,-0.043048,-0.018001,-0.014584,-0.000918,0.000000,-0.017706,-0.014186,-0.007825,...,0.003148,0.0,0.024886,-0.007139,-0.089551,-0.073484,0.0,0.001634,-0.022793,0.0
2016-01-07,-0.026577,0.0,0.008977,-0.049544,0.031124,-0.041487,-0.023790,-0.032033,-0.020222,-0.039423,...,-0.033222,0.0,-0.026207,-0.040240,-0.057385,-0.059489,0.0,-0.000324,-0.028658,0.0
2016-01-08,0.013059,0.0,-0.076707,-0.027028,0.013859,0.012497,0.000000,-0.002064,-0.019568,0.019005,...,0.042802,0.0,0.002299,0.016059,0.026059,-0.033909,0.0,-0.047958,-0.004820,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016-12-23,0.011374,0.0,0.012770,0.015452,0.002058,0.028773,0.028466,0.023968,0.020928,0.005536,...,-0.007260,0.0,0.000426,-0.001213,-0.002512,-0.007483,0.0,0.012328,0.026989,0.0
2016-12-26,0.011255,0.0,-0.005930,0.020596,-0.014373,0.008878,0.015694,0.020484,0.018309,0.002934,...,-0.015609,0.0,0.002086,0.001519,0.012655,0.031760,0.0,0.007919,-0.008767,0.0
2016-12-27,-0.004325,0.0,0.020128,-0.017107,-0.000415,0.013874,0.007006,0.002505,-0.009704,-0.015366,...,0.005943,0.0,0.001671,0.005453,0.024994,-0.005000,0.0,-0.001398,0.016312,0.0


## EXCESSO DOS ANOS e SIGMA (MAtriz de covariancia)

##### EXCESSO PARA TODOS

In [22]:
dict_sigma = {}
dict_excesso = {}
for an in anos:
    ano = an.split("-")[0]
    if ano == '2015':
        pass
    else:
        try:
            print(f"=============== \n EXCESSO {ano}\n ============")
            print("Atualização, Ano: ",ano)
            df = f'df_ativos_{ano}'
            df_f = pd.DataFrame(eval(df))
            slc = selic_d[selic_d.index.isin(df_f.index)]
            slc['valor_diario'] = slc['valor_diario']/100

            print(f"Tamanho DF de {ano}: ", len(df_f))
            print(f"Tamanho Selic: ", len(slc['valor_diario']))
            ano = int(ano)
            dict_excesso[ano] = df_f.sub(slc['valor_diario'],axis=0)
            print("tamanho final do Excesso: ",len(dict_excesso[ano]))

            print(f"============\n SIGMA {ano}\n===========")            
            dict_sigma[ano] = dict_excesso[ano].cov()
            print('Tamanho final do SIGMA: ',len(dict_sigma[ano]))

        except Exception as e:
            print(e)
            print("ERror")

 EXCESSO 2025
Atualização, Ano:  2025
Tamanho DF de 2025:  250
Tamanho Selic:  250
tamanho final do Excesso:  250
 SIGMA 2025
Tamanho final do SIGMA:  78
 EXCESSO 2024
Atualização, Ano:  2024
Tamanho DF de 2024:  251
Tamanho Selic:  251
tamanho final do Excesso:  251
 SIGMA 2024
Tamanho final do SIGMA:  78
 EXCESSO 2023
Atualização, Ano:  2023
Tamanho DF de 2023:  248
Tamanho Selic:  248
tamanho final do Excesso:  248
 SIGMA 2023
Tamanho final do SIGMA:  78
 EXCESSO 2022
Atualização, Ano:  2022
Tamanho DF de 2022:  250
Tamanho Selic:  250
tamanho final do Excesso:  250
 SIGMA 2022
Tamanho final do SIGMA:  78
 EXCESSO 2021
Atualização, Ano:  2021
Tamanho DF de 2021:  247
Tamanho Selic:  247
tamanho final do Excesso:  247
 SIGMA 2021
Tamanho final do SIGMA:  78
 EXCESSO 2020
Atualização, Ano:  2020
Tamanho DF de 2020:  248
Tamanho Selic:  248
tamanho final do Excesso:  248
 SIGMA 2020
Tamanho final do SIGMA:  78
 EXCESSO 2019
Atualização, Ano:  2019
Tamanho DF de 2019:  248
Tamanho Selic

In [23]:
exc_2016 = df_ativos_2016.sub(selic_d_2016['valor_diario'], axis=0)
sharpe_2016 = exc_2016.mean() / df_ativos_2016.std()   # Series: index=ticker


#### Dados da Magic Formula (SCORE)

In [24]:


# y1 = []
# y2 = []
# for a in anos[:-1]:
#     y1.append(a)
# for a in anos[:-1]:
#     y2.append(a.split('-')[0])

# print("=-"*48)
# print("Anos totais: ",y)
# print("=-"*48)

dict_score_todos = []
dict_score = {}
for i, ativo in enumerate(df_ativos_2016.columns.tolist()):
        if ativo == lista_mf_check[i]['ativo']:
            for ano in anos:
                df_temp = lista_mf_check[i]['data'].query("endDate in @ano")
                score_anual = (0.5*df_temp['ROC'] + 0.5*df_temp['EY']).sum()
                dict_score[(ano, ativo)] = score_anual
                print("feito dict do ativo: ",ativo,ano,score_anual)
        else:
            print(f" ativo: {ativo}")
            print(f" ativo_mf: {lista_mf_check[i]['ativo']}")
df_scores = pd.Series(dict_score).unstack()  # index=anos, columns=ativos
df_scores = df_scores.replace([np.inf, -np.inf],0)    

dict_score_todos.append(df_scores)  


feito dict do ativo:  ABEV3 2025-12-31 0.5075949601583762
feito dict do ativo:  ABEV3 2024-12-31 0.38261468462358406
feito dict do ativo:  ABEV3 2023-12-31 0.4707780476577104
feito dict do ativo:  ABEV3 2022-12-31 0.3698561983003272
feito dict do ativo:  ABEV3 2021-12-31 0.33762441702482754
feito dict do ativo:  ABEV3 2020-12-31 0.3419377467430017
feito dict do ativo:  ABEV3 2019-12-31 0.3554827706790753
feito dict do ativo:  ABEV3 2018-12-31 0.4588246122822893
feito dict do ativo:  ABEV3 2017-12-31 0.587594465523369
feito dict do ativo:  ABEV3 2016-12-31 0.6475658938442502
feito dict do ativo:  ABEV3 2015-12-31 0.5948621071274975
feito dict do ativo:  ALOS3 2025-12-31 0.39483852927400404
feito dict do ativo:  ALOS3 2024-12-31 0.3900179383466277
feito dict do ativo:  ALOS3 2023-12-31 1.655507111089378
feito dict do ativo:  ALOS3 2022-12-31 0.0
feito dict do ativo:  ALOS3 2021-12-31 0.0
feito dict do ativo:  ALOS3 2020-12-31 0.0
feito dict do ativo:  ALOS3 2019-12-31 0.0
feito dict do a

In [25]:
dict_score_todos[0].loc['2016-12-31'].rank(axis=0, pct=True)

ABEV3    0.897436
ALOS3    0.243590
ANIM3    0.538462
AXIA3    0.833333
AZZA3    0.666667
           ...   
VALE3    0.551282
VBBR3    0.243590
VIVT3    0.730769
WEGE3    0.576923
YDUQ3    0.243590
Name: 2016-12-31, Length: 78, dtype: float64

In [26]:
score_todos_anos = dict_score_todos[0].rank(axis=1,pct=True)

In [27]:
score_todos_anos

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
2015-12-31,0.884615,0.307692,0.641026,0.064103,0.692308,0.897436,0.307692,0.307692,0.307692,0.012821,...,0.948718,0.307692,0.769231,0.730769,0.051282,0.076923,0.307692,0.666667,0.564103,0.307692
2016-12-31,0.897436,0.243590,0.538462,0.833333,0.666667,0.602564,0.243590,0.243590,0.243590,1.000000,...,0.923077,0.243590,0.782051,0.679487,0.089744,0.551282,0.243590,0.730769,0.576923,0.243590
2017-12-31,0.910256,0.185897,0.551282,0.461538,0.628205,0.858974,0.185897,0.185897,0.185897,0.961538,...,0.820513,0.602564,0.576923,0.564103,0.371795,0.589744,0.185897,0.666667,0.474359,0.185897
2018-12-31,0.871795,0.173077,0.320513,0.756410,0.576923,0.884615,0.173077,0.173077,0.173077,0.961538,...,0.897436,0.538462,0.551282,0.346154,0.333333,0.602564,0.173077,0.730769,0.512821,0.173077
2019-12-31,0.833333,0.198718,0.448718,0.551282,0.679487,0.923077,0.198718,0.198718,0.198718,1.000000,...,0.807692,0.525641,0.512821,0.320513,0.333333,0.294872,0.198718,0.666667,0.602564,0.730769
2020-12-31,0.782051,0.179487,0.269231,0.525641,0.730769,0.794872,0.179487,0.179487,0.179487,0.961538,...,0.935897,0.500000,0.641026,0.358974,0.589744,0.602564,0.179487,0.564103,0.538462,0.307692
2021-12-31,0.717949,0.102564,0.269231,0.538462,0.794872,0.756410,0.102564,0.102564,0.102564,0.974359,...,0.987179,0.025641,0.384615,0.243590,0.807692,0.782051,0.410256,0.448718,0.487179,0.320513
2022-12-31,0.756410,0.115385,0.371795,0.307692,0.666667,0.743590,0.115385,0.115385,0.115385,0.987179,...,0.833333,0.025641,0.384615,0.615385,0.538462,0.717949,0.551282,0.448718,0.564103,0.410256
2023-12-31,0.794872,0.935897,0.538462,0.448718,0.628205,0.769231,0.096154,0.096154,0.096154,0.987179,...,0.833333,0.166667,0.384615,0.589744,0.179487,0.564103,0.730769,0.423077,0.576923,0.461538
2024-12-31,0.782051,0.794872,0.641026,0.615385,0.217949,0.820513,0.108974,0.108974,0.108974,1.000000,...,0.948718,0.435897,0.679487,0.705128,0.166667,0.538462,0.769231,0.371795,0.589744,0.564103


In [28]:
index_novo = []
for indexx in score_todos_anos.index:
    novo = indexx.split('-')[0]
    index_novo.append(novo)
print(index_novo)

['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [29]:
score_todos_anos.index=  index_novo

In [30]:
score_todos_anos

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
2015,0.884615,0.307692,0.641026,0.064103,0.692308,0.897436,0.307692,0.307692,0.307692,0.012821,...,0.948718,0.307692,0.769231,0.730769,0.051282,0.076923,0.307692,0.666667,0.564103,0.307692
2016,0.897436,0.243590,0.538462,0.833333,0.666667,0.602564,0.243590,0.243590,0.243590,1.000000,...,0.923077,0.243590,0.782051,0.679487,0.089744,0.551282,0.243590,0.730769,0.576923,0.243590
2017,0.910256,0.185897,0.551282,0.461538,0.628205,0.858974,0.185897,0.185897,0.185897,0.961538,...,0.820513,0.602564,0.576923,0.564103,0.371795,0.589744,0.185897,0.666667,0.474359,0.185897
2018,0.871795,0.173077,0.320513,0.756410,0.576923,0.884615,0.173077,0.173077,0.173077,0.961538,...,0.897436,0.538462,0.551282,0.346154,0.333333,0.602564,0.173077,0.730769,0.512821,0.173077
2019,0.833333,0.198718,0.448718,0.551282,0.679487,0.923077,0.198718,0.198718,0.198718,1.000000,...,0.807692,0.525641,0.512821,0.320513,0.333333,0.294872,0.198718,0.666667,0.602564,0.730769
2020,0.782051,0.179487,0.269231,0.525641,0.730769,0.794872,0.179487,0.179487,0.179487,0.961538,...,0.935897,0.500000,0.641026,0.358974,0.589744,0.602564,0.179487,0.564103,0.538462,0.307692
2021,0.717949,0.102564,0.269231,0.538462,0.794872,0.756410,0.102564,0.102564,0.102564,0.974359,...,0.987179,0.025641,0.384615,0.243590,0.807692,0.782051,0.410256,0.448718,0.487179,0.320513
2022,0.756410,0.115385,0.371795,0.307692,0.666667,0.743590,0.115385,0.115385,0.115385,0.987179,...,0.833333,0.025641,0.384615,0.615385,0.538462,0.717949,0.551282,0.448718,0.564103,0.410256
2023,0.794872,0.935897,0.538462,0.448718,0.628205,0.769231,0.096154,0.096154,0.096154,0.987179,...,0.833333,0.166667,0.384615,0.589744,0.179487,0.564103,0.730769,0.423077,0.576923,0.461538
2024,0.782051,0.794872,0.641026,0.615385,0.217949,0.820513,0.108974,0.108974,0.108974,1.000000,...,0.948718,0.435897,0.679487,0.705128,0.166667,0.538462,0.769231,0.371795,0.589744,0.564103


## Hiperparâmetros

In [31]:
vb_cardinalidade_max = 20
vb_cardinalidade_min = 20
vb_peso_maximo = 0.15
vb_peso_minimo = 0.02
vb_theta = 0.30

### Fazendo otimização ano a ano e comparando com o proximo ano

In [36]:
anos = anos[::-1]
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [38]:
from pydoc import text


y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])



print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass

    print("-"*15)
    print("Começando Modelo do ano: ", ano)
    print("="*6)

    try:
        score_usado = score_todos_anos[score_todos_anos.index.isin([ano])]
        # print(score_usado)
        print("Score Atualizado")
        

        ano_um = int(ano)+1
        df_usado = f'df_ativos_{ano_um}'
        retorno_usado = pd.DataFrame(eval(df_usado))
        retorno_usado = retorno_usado[lista_ativos_finais]
        print("Retornos atualizados")

        excesso_usado = dict_excesso[ano_um]
        print("Excessos Atualizados")
        
        sigma_usado = dict_sigma[ano_um]
        print("Sigmas Atualizados")
        
        print("-----")
    except Exception as e:
        print("ERRRRRRRRRRRROR")
        print(e)

    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print("# ------ CRIAÇÃO DO MODELO")
    print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",df_usado)
    print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = lista_ativos_finais)
    model.ativos = pyo.RangeSet(0, len(lista_ativos_finais)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.theta = pyo.Param(initialize=vb_theta)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score_usado.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos, bounds=(0,1))
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    model.s = pyo.Param(initialize = 2, mutable=True)
    model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
        retorno_esperado = model.theta * sum(
            model.retornos_ativos[dia, a] * model.x[a] for a in model.ativos for dia in model.dias
        )
        # retorno_esperado = model.theta * model.r
        # A dúvida de qual retorno usar (Retorno BRUTO ou Excesso (retorno - selic))

        score_total = (1 - model.theta) * sum(
            model.x[a] * model.score[a] for a in model.ativos
        )
        return retorno_esperado + score_total
    model.obj1 = pyo.Objective(rule=func_objetivo_1, sense=pyo.maximize)

    #=============================
    # RESTRIÇÕES
    #=============================

    def def_r(model):
        return model.r == sum(model.excesso[a]*model.x[a]for a in model.ativos)
    model.const_def_r = pyo.Constraint(rule=def_r)

    ## modelos de Programação de Cone de Segunda Ordem (SOCP) 
    ## e Programação Quadrática com Restrições (QCP)
    def cone(model):
        return model.r**2 >= model.s**2 * sum(model.x[a]*model.sigma[a,b]*model.x[b] for a in model.ativos for b in model.ativos)
    model.constr_cone = pyo.Constraint(rule=cone)

    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    #peso minimo por acao
    def rule_peso_minimo(model, a):
        return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)


    def cardinalidade_min(model):
        return sum(
            model.y[a] for a in model.ativos
            ) >= model.cardinalidade_valor_min
    model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')

    s_lo = 0.01        # <-- o Sharpe DIÁRIO 
    s_hi = 1   # teto: 2x o melhor ativo individual
    tol  = 0.001
    print("MODEL.S.VALUE => ",model.s.value)
    melhor_pesos = None
    historico = []
    carteiras_criadas = []
    # print(f"Valor a ser batido com S_LO: {s_lo} e S_HI: {s_hi} ... Diferença: {s_hi-s_lo}")
    print(f"Começando o WHILE do ano {ano}")
    while s_hi - s_lo > tol:
        s_a_ser_usado = 0.5 * (s_lo + s_hi)
        # print(s_a_ser_usado)
        model.s.value = s_a_ser_usado
        print("="*18)
        # print("Objetivo: (É o S lá da restrição de sharpe): ",s_a_ser_usado)
        res = opt.solve(model, load_solutions=False, tee=False)
        tc = res.solver.termination_condition
        print(f"Modelo Resolveu....Condição: ",tc)
        if tc == pyo.TerminationCondition.optimal:
            model.solutions.load_from(res)
            melhor_pesos = {a: pyo.value(model.x[a]) for a in model.ativos}
            # print({x:i for x,i in melhor_pesos.items() if i>0})
            s_lo = model.s.value
            print(f"Valor encontrado para o Sharpe: ",s_lo)
            # print(f"Valor FO: ",pyo.value(model.obj1))
            # print(f"Atualização de Valores para próxima rodagem: TC {tc} com S_LO: {s_lo} e S_HI: {s_hi} ... Diferença: {s_hi-s_lo}")
            # print("="*28)
            # for x,i in melhor_pesos.items():
            #     if i>0:

            #         dc = {
            #         'ativos_selecionados':melhor_pesos,
            #         'sharpe':pyo.value(model.s),
            #             }
            historico.append((s_a_ser_usado, 'viavel'))
            # carteiras_criadas.append(dc)
        elif tc in (pyo.TerminationCondition.infeasible,
                    pyo.TerminationCondition.infeasibleOrUnbounded):
            s_hi = s_a_ser_usado
            print(f"Atualização do S_HI : {s_hi}")
            print("="*28)

            historico.append((s_a_ser_usado, 'inviavel'))
        else:
            print(f'status inesperado em s={s_a_ser_usado:.4f}: {tc}')
            pass
    print("SAIU DO WHILE")
    print(f'Sharpe máximo (diário) ≈ {s_a_ser_usado:.4f}  |  anualizado ≈ {s_a_ser_usado*np.sqrt(252):.2f}')
    # print(historico)
    carteiras_anuais[int(ano)] = {
        'pesos':  melhor_pesos,
        'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass

    ## OQ ESTÀ FALTANDO?
    ## FAZER O SIGMA PARA TODOS


=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Modelo Antigo deletado 
 Iniciando Novo
---------------
Começando Modelo do ano:  2015
Score Atualizado
Retornos atualizados
Excessos Atualizados
Sigmas Atualizados
-----
# ------ CRIAÇÃO DO MODELO
## UTILIZANDO SCORE DO ANO DE:  2015
## UTILIZANDO DADOS DE RETORNO DE:  df_ativos_2016
## UTILIZANDO EXCESSO DO ANO DE:  2016
## UTILIZANDO SIGMAS DO ANO DE:  2016
MODEL.S.VALUE =>  2
Começando o WHILE do ano 2015
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.505
Modelo Resolveu....Condição:  optimal
Valor encontrado para o Sharpe:  0.2575
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.38125
Modelo Resolveu....Condição:  infeasible
Atual

In [40]:
ra = range(0,len(lista_ativos_finais))

for an in anos:
    an = int(an.split('-')[0])
    print(an)
    print([0 if carteiras_anuais[an]['pesos'][a] <0.02 else carteiras_anuais[an]['pesos'][a] for a in ra])

2015
[0, 0.020000010405344763, 0, 0.10898229009347347, 0, 0, 0, 0, 0, 0, 0, 0, 0.020000007412741116, 0, 0, 0, 0, 0.020000037388439012, 0, 0, 0.09782814079935771, 0, 0, 0.02, 0, 0, 0, 0, 0, 0, 0.10798867112242323, 0, 0, 0.14999999543572304, 0, 0, 0, 0.02, 0.02, 0, 0, 0, 0, 0, 0, 0, 0.02, 0.10520092266857674, 0, 0.02, 0, 0, 0, 0, 0.020000010823238434, 0, 0, 0.14999989893711208, 0, 0, 0, 0, 0, 0, 0, 0.02, 0.020000010137187583, 0, 0, 0.02, 0, 0, 0, 0, 0.02, 0, 0, 0.020000010407959855]
2016
[0, 0, 0.06754428034093454, 0, 0.031608010846826746, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.08189000679144562, 0, 0, 0, 0, 0, 0, 0, 0.02000002619991728, 0, 0, 0.04833983489834474, 0, 0, 0, 0, 0.11043650244873764, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.149999977925695, 0.02000000392701874, 0, 0, 0, 0.06504190071744884, 0, 0, 0, 0.11428050046663966, 0, 0, 0.11085882730125977, 0, 0, 0, 0, 0, 0, 0, 0, 0.020000003240238318, 0]
2017
[0, 0.020000314383101545, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [41]:
data = []
for carteira in carteiras_anuais:
    # print(carteiras_anuais[carteira])
    # print(len(carteiras_anuais[carteira]['pesos']))
    # print(len(lista_ativos_finais))
    # print(len(ra))
    final = pd.DataFrame({
        'peso_ativo': [carteiras_anuais[carteira]['pesos'][a] for a in ra],
        'ativado': [0 if carteiras_anuais[carteira]['pesos'][a] < vb_peso_minimo else 1 for a in ra],
    }, index=lista_ativos_finais)
    # print(len(final))
    df_final = final[final['ativado']!=0]
    data.append(df_final)

In [42]:
np.round(data[0].sort_values(by='peso_ativo', ascending=False)['peso_ativo'],4)

FLRY3     0.1500
RADL3     0.1500
AXIA3     0.1090
ENGI11    0.1080
MGLU3     0.1052
CSMG3     0.0978
CPFE3     0.0200
POMO4     0.0200
YDUQ3     0.0200
ALOS3     0.0200
SMTO3     0.0200
BRAP4     0.0200
MDNE3     0.0200
IRBR3     0.0200
IGTI11    0.0200
CXSE3     0.0200
SMFT3     0.0200
MOVI3     0.0200
TEND3     0.0200
VBBR3     0.0200
Name: peso_ativo, dtype: float64